## Fine-Tuning Neural Network Hyperparameters with Optuna

In [7]:
import torch
import torch.nn as nn
import torchvision 
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)

test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000]
)

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [9]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )
    
    def forward(self, X):
        return self.mlp(X)

In [10]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

Let’s tune the learning rate and the number of neurons in the hidden layers (for simplicity, we will use the same number of neurons in both hidden layers). First, we need to define a function that Optuna will call many times to perform hyperparameter tuning: this function
must take a Trial object and use it to ask Optuna for hyperparameter values, and then use these hyperparameter values to build and train a model. Finally, the function must evaluate the model (typically on the validation set) and return the metric.

In [11]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            preds = torch.argmax(y_pred, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

def objective(trail):
    learning_rate = trail.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trail.suggest_int("n_hidden", 20, 300)

    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    n_epochs = 20

    train(model, optimizer, xentropy, train_loader, n_epochs)
    valid_acc = evaluate(model, valid_loader)

    return valid_acc

In [17]:
import optuna

torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=3)

d:\programming\machine_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-03 19:00:07,356] A new study created in memory with name: no-name-c5dc5e60-60c3-4f56-8f61-3026038caf0b


Epoch 1/20, Loss: 2.2769
Epoch 2/20, Loss: 2.2093
Epoch 3/20, Loss: 2.1164
Epoch 4/20, Loss: 1.9776
Epoch 5/20, Loss: 1.7867
Epoch 6/20, Loss: 1.5775
Epoch 7/20, Loss: 1.3979
Epoch 8/20, Loss: 1.2605
Epoch 9/20, Loss: 1.1573
Epoch 10/20, Loss: 1.0782
Epoch 11/20, Loss: 1.0162
Epoch 12/20, Loss: 0.9665
Epoch 13/20, Loss: 0.9258
Epoch 14/20, Loss: 0.8918
Epoch 15/20, Loss: 0.8629
Epoch 16/20, Loss: 0.8382
Epoch 17/20, Loss: 0.8165
Epoch 18/20, Loss: 0.7974
Epoch 19/20, Loss: 0.7803
Epoch 20/20, Loss: 0.7649


[I 2026-07-03 19:10:02,586] Trial 0 finished with value: 0.7102 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7102.


Epoch 1/20, Loss: 1.1392
Epoch 2/20, Loss: 0.6219
Epoch 3/20, Loss: 0.5251
Epoch 4/20, Loss: 0.4826
Epoch 5/20, Loss: 0.4571
Epoch 6/20, Loss: 0.4405
Epoch 7/20, Loss: 0.4239
Epoch 8/20, Loss: 0.4121
Epoch 9/20, Loss: 0.4020
Epoch 10/20, Loss: 0.3921
Epoch 11/20, Loss: 0.3839
Epoch 12/20, Loss: 0.3743
Epoch 13/20, Loss: 0.3665
Epoch 14/20, Loss: 0.3595
Epoch 15/20, Loss: 0.3521
Epoch 16/20, Loss: 0.3458
Epoch 17/20, Loss: 0.3400
Epoch 18/20, Loss: 0.3342
Epoch 19/20, Loss: 0.3280
Epoch 20/20, Loss: 0.3227


[I 2026-07-03 19:19:47,082] Trial 1 finished with value: 0.87 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.87.


Epoch 1/20, Loss: 2.3091
Epoch 2/20, Loss: 2.3005
Epoch 3/20, Loss: 2.2929
Epoch 4/20, Loss: 2.2859
Epoch 5/20, Loss: 2.2792
Epoch 6/20, Loss: 2.2726
Epoch 7/20, Loss: 2.2662
Epoch 8/20, Loss: 2.2598
Epoch 9/20, Loss: 2.2534
Epoch 10/20, Loss: 2.2469
Epoch 11/20, Loss: 2.2402
Epoch 12/20, Loss: 2.2332
Epoch 13/20, Loss: 2.2256
Epoch 14/20, Loss: 2.2177
Epoch 15/20, Loss: 2.2094
Epoch 16/20, Loss: 2.2006
Epoch 17/20, Loss: 2.1913
Epoch 18/20, Loss: 2.1813
Epoch 19/20, Loss: 2.1705
Epoch 20/20, Loss: 2.1592


[I 2026-07-03 19:29:46,593] Trial 2 finished with value: 0.2428 and parameters: {'learning_rate': 4.207988669606632e-05, 'n_hidden': 63}. Best is trial 1 with value: 0.87.


In [18]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.008471801418819975, 'n_hidden': 188}
0.87


Optuna comes with several Pruner classes that can detect and prune bad trials. For example, the MedianPruner will prune trials whose performance is below the median performance, at regular intervals during training. 

In [30]:
pruner = optuna.pruners.MedianPruner(n_startup_trials=1, n_warmup_steps=3, interval_steps=1)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

[I 2026-07-03 21:01:00,823] A new study created in memory with name: no-name-1248fafe-c3b6-462d-8acb-70cb39b34a8f


Now we need a slightly modified objective function in which after each epoch, it checks validation accuracy and reports it to optuna of the current validation accuracy and epoch so it can determine whether the trail should be pruned.

In [31]:
def objective(trail):
    learning_rate = trail.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trail.suggest_int("n_hidden", 20, 300)

    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    n_epochs = 20

    # instead of the train function call, we paste the modified train function
     
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

        # added part for pruning 
        
        valid_acc = evaluate(model, valid_loader)
        trail.report(valid_acc, epoch)
        if trail.should_prune():
            raise optuna.TrialPruned()

    return valid_acc

In [32]:
study.optimize(objective, n_trials=5)

Epoch 1/20, Loss: 2.0537
Epoch 2/20, Loss: 1.1728
Epoch 3/20, Loss: 0.8514
Epoch 4/20, Loss: 0.7498
Epoch 5/20, Loss: 0.6907
Epoch 6/20, Loss: 0.6455
Epoch 7/20, Loss: 0.6091
Epoch 8/20, Loss: 0.5794
Epoch 9/20, Loss: 0.5543
Epoch 10/20, Loss: 0.5337
Epoch 11/20, Loss: 0.5153
Epoch 12/20, Loss: 0.5009
Epoch 13/20, Loss: 0.4889
Epoch 14/20, Loss: 0.4785
Epoch 15/20, Loss: 0.4696
Epoch 16/20, Loss: 0.4622
Epoch 17/20, Loss: 0.4565
Epoch 18/20, Loss: 0.4503
Epoch 19/20, Loss: 0.4449
Epoch 20/20, Loss: 0.4406


[I 2026-07-03 21:15:23,779] Trial 0 finished with value: 0.833 and parameters: {'learning_rate': 0.00234238498471129, 'n_hidden': 33}. Best is trial 0 with value: 0.833.


Epoch 1/20, Loss: 1.9280
Epoch 2/20, Loss: 1.0509
Epoch 3/20, Loss: 0.7943
Epoch 4/20, Loss: 0.6946
Epoch 5/20, Loss: 0.6295
Epoch 6/20, Loss: 0.5820
Epoch 7/20, Loss: 0.5480
Epoch 8/20, Loss: 0.5227
Epoch 9/20, Loss: 0.5047
Epoch 10/20, Loss: 0.4902
Epoch 11/20, Loss: 0.4789
Epoch 12/20, Loss: 0.4701
Epoch 13/20, Loss: 0.4619
Epoch 14/20, Loss: 0.4549
Epoch 15/20, Loss: 0.4486
Epoch 16/20, Loss: 0.4437
Epoch 17/20, Loss: 0.4391
Epoch 18/20, Loss: 0.4346
Epoch 19/20, Loss: 0.4296
Epoch 20/20, Loss: 0.4259


[I 2026-07-03 21:24:22,471] Trial 1 finished with value: 0.8418 and parameters: {'learning_rate': 0.0026926469100861782, 'n_hidden': 67}. Best is trial 1 with value: 0.8418.


Epoch 1/20, Loss: 2.3010
Epoch 2/20, Loss: 2.2975
Epoch 3/20, Loss: 2.2939
Epoch 4/20, Loss: 2.2904


[I 2026-07-03 21:26:13,587] Trial 2 pruned. 


Epoch 1/20, Loss: 0.6209
Epoch 2/20, Loss: 0.4156
Epoch 3/20, Loss: 0.3683
Epoch 4/20, Loss: 0.3405
Epoch 5/20, Loss: 0.3193
Epoch 6/20, Loss: 0.3018
Epoch 7/20, Loss: 0.2903
Epoch 8/20, Loss: 0.2770
Epoch 9/20, Loss: 0.2664
Epoch 10/20, Loss: 0.2567
Epoch 11/20, Loss: 0.2469
Epoch 12/20, Loss: 0.2396
Epoch 13/20, Loss: 0.2298
Epoch 14/20, Loss: 0.2240
Epoch 15/20, Loss: 0.2176
Epoch 16/20, Loss: 0.2105
Epoch 17/20, Loss: 0.2045
Epoch 18/20, Loss: 0.1979
Epoch 19/20, Loss: 0.1924
Epoch 20/20, Loss: 0.1869


[I 2026-07-03 21:35:55,990] Trial 3 finished with value: 0.8896 and parameters: {'learning_rate': 0.07286653737491042, 'n_hidden': 247}. Best is trial 3 with value: 0.8896.


Epoch 1/20, Loss: 2.2917
Epoch 2/20, Loss: 2.2610
Epoch 3/20, Loss: 2.2284
Epoch 4/20, Loss: 2.1887


[I 2026-07-03 21:37:50,565] Trial 4 pruned. 


In [33]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.07286653737491042, 'n_hidden': 247}
0.8896


Lets train the model with these hyperparameters on both train and valid data and test it.

In [12]:
train_valid_loader = DataLoader(train_and_valid_data, batch_size=32, shuffle=True)

In [13]:
learning_rate = 0.0728
n_hidden = 247

model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                        n_hidden2=n_hidden, n_classes=10).to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
xentropy = nn.CrossEntropyLoss()
n_epochs = 20

train(model, optimizer, xentropy, train_valid_loader, n_epochs)

Epoch 1/20, Loss: 0.6061
Epoch 2/20, Loss: 0.4085
Epoch 3/20, Loss: 0.3638
Epoch 4/20, Loss: 0.3371
Epoch 5/20, Loss: 0.3163
Epoch 6/20, Loss: 0.2998
Epoch 7/20, Loss: 0.2872
Epoch 8/20, Loss: 0.2748
Epoch 9/20, Loss: 0.2644
Epoch 10/20, Loss: 0.2533
Epoch 11/20, Loss: 0.2450
Epoch 12/20, Loss: 0.2374
Epoch 13/20, Loss: 0.2297
Epoch 14/20, Loss: 0.2230
Epoch 15/20, Loss: 0.2156
Epoch 16/20, Loss: 0.2100
Epoch 17/20, Loss: 0.2036
Epoch 18/20, Loss: 0.1975
Epoch 19/20, Loss: 0.1915
Epoch 20/20, Loss: 0.1859


In [14]:
print(evaluate(model, train_valid_loader))
print(evaluate(model, test_loader))

0.9312333333333334
0.8888


## Saving and Loading PyTorch Models

In [74]:
torch.save(model, "fashion_mnist.pt")

In [75]:
loaded_model = torch.load("fashion_mnist.pt", weights_only=False)

In [76]:
X_new, y_new = next(iter(test_loader))
X_new = X_new[:3].to(device)
y_new = y_new[:3].to(device)

loaded_model.eval()
y_pred_logits = loaded_model(X_new)

y_pred = torch.argmax(y_pred_logits, dim=1)

print(y_pred)
print(y_new)

tensor([9, 2, 1], device='cuda:0')
tensor([9, 2, 1], device='cuda:0')


This method is easy but has issues with security and instability with versions.  
It is recommended to save and load the model weights only, rather than the full model object.

In [77]:
torch.save(model.state_dict(), "fashion_mnist_weights.pt")

In [78]:
loaded_weights = torch.load("fashion_mnist_weights.pt", weights_only=True)

new_model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                            n_hidden2=n_hidden, n_classes=10).to(device)
new_model.load_state_dict(loaded_weights)

<All keys matched successfully>

This only works if you are able to create the exact same
model architecture before loading the state dictionary. For this, you need to know the number of layers, the number of neurons per layer, and so on. It’s a good idea to save this information along with the state dictionary

In [79]:
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {
        "n_inputs": 1 * 28 * 28,
        "n_hidden1": n_hidden,
        "n_hidden2": n_hidden,
        "n_classes": 10
    }
}
torch.save(model_data, "fashion_mnist_data.pt")

In [80]:
loaded_data = torch.load("fashion_mnist_data.pt", weights_only=True)
new_model = ImageClassifier(**loaded_data["model_hyperparameters"])
new_model.load_state_dict(loaded_data["model_state_dict"])

<All keys matched successfully>

There is yet another way to save and load your model: by first converting it to TorchScript. This also makes it possible to speed up your model’s inference.

## Compiling and Optimizing a PyTorch Model

PyTorch comes with a very nice feature: it can automatically convert your model’s code to TorchScript, which you can think of as a statically typed subset of Python.  
There are two main benefits:  
1. TorchScript code can be compiled and optimized to produce significantly faster models.  
2. TorchScript can be serialized, saved to disk, and then loaded and
executed in Python or in a C++ environment using the LibTorch library. This makes it possible to run PyTorch models on a wide range of devices, including embedded devices.

There are two ways to convert a PyTorch model to TorchScript. The first way is called tracing. PyTorch just runs your model with some sample data, logs every operation that takes place, and then converts this log to TorchScript.

In [81]:
torchscript_model = torch.jit.trace(model, X_new)

This generally works well with static models whose forward() method doesn’t use conditionals or loops. However, if you try to trace a model that includes an if or match statement, then only the branch that is actually executed will be captured by TorchScript, which is generally not what you want. Similarly, if you use tracing with
a model that contains a loop, then the TorchScript code will contain one copy of the operations within that loop for each iteration that was actually executed. Again, not what you generally want.

For such dynamic models, you will probably want to try another approach named scripting. In this case, PyTorch actually parses your Python code directly and converts it to TorchScript. 

In [85]:
torchscript_model = torch.jit.script(model)

Regardless of whether you use tracing or scripting to produce your TorchScript model, you can then further optimize it:

In [86]:
optimized_model = torch.jit.optimize_for_inference(torchscript_model)

TorchScript models can only be used for inference, not for training, since the TorchScript environment doesn’t support gradient tracking or parameter updates.  
Finally, you can save a TorchScript model using its save() method:

In [87]:
torchscript_model.save('fashion_mnist_torchscript.pt')

And then load it using the torch.jit.load() function:

In [88]:
loaded_torchscript_model = torch.jit.load('fashion_mnist_torchscript.pt')

In [89]:
compiled_model = torch.compile(model)